# Install Library

In [1]:
!pip install transformers
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import Library

In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer
from google.colab import drive

# Connect to Drive

In [4]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Path folder tempat file CSV berada
folder_path = "/content/drive/MyDrive/File/CODES/Summarize/5_Summary_Suara_Nasional_50_Base1/Sum_Suara_Nasional_50_Base1"

# Mengambil semua file dengan ekstensi .csv dalam folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith(".csv")]

# Membaca dan menggabungkan semua file
dataframes = []
for file_name in csv_files:
    file_path = os.path.join(folder_path, file_name)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Menggabungkan semua DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)

In [6]:
# Menyimpan hasil penggabungan ke file baru
output_path = os.path.join(folder_path, "/content/drive/MyDrive/File/CODES/Similarity/5_Similarity_Suara_Nasional_Base1/suara_nas_sum_50_base1_concat.csv")
combined_df.to_csv(output_path, index=False)

In [7]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6102 entries, 0 to 6101
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         6102 non-null   object
 1   category      6102 non-null   object
 2   publish_date  6102 non-null   object
 3   author        6102 non-null   object
 4   article_url   6102 non-null   object
 5   content       6102 non-null   object
 6   summary       6102 non-null   object
dtypes: object(7)
memory usage: 333.8+ KB


In [8]:
combined_df.head()

,title,category,publish_date,author,article_url,content,summary
0,Jokowi Beberkan 'Love-Hate' dengan Surya Paloh...,Nasional,2024-08-26,Chandra Iswinarno,https://www.suara.com/news/2024/08/26/070000/j...,Presiden Joko Widodo atau Jokowi membeberkan b...,Presiden Joko Widodo atau Jokowi membeberkan b...
1,"Ucap Terima Kasih, Surya Paloh Akui Belajar da...",Nasional,2024-08-25,Agung Sandy Lesmana,https://www.suara.com/news/2024/08/25/224500/u...,Ketua Umum Partai NasDem Surya Paloh mengaku b...,Ketua Umum Partai NasDem Surya Paloh mengaku b...
2,"Di Depan Jokowi, Surya Paloh: Bukan Cuma Masal...",Nasional,2024-08-25,Agung Sandy Lesmana,https://www.suara.com/news/2024/08/25/214434/d...,Ketua Umum Partai NasDem Surya Paloh menegaska...,Ketua Umum Partai NasDem Surya Paloh menegaska...
3,"Jokowi Muncul di Kongres III NasDem Malam Ini,...",Nasional,2024-08-25,Agung Sandy Lesmana,https://www.suara.com/news/2024/08/25/194135/j...,Presiden Joko Widodo alias Jokowi menghadiri K...,Presiden Joko Widodo alias Jokowi menghadiri K...
4,"Sebut Pemilu 2024 Paling Brutal, Cak Imin Usul...",Nasional,2024-08-25,Bangun Santoso,https://www.suara.com/news/2024/08/25/184735/s...,Ketua Umum Dewan Pimpinan Pusat Partai Kebangk...,Ketua Umum Dewan Pimpinan Pusat Partai Kebangk...


In [9]:
nan_summary = combined_df[combined_df['summary'].isna()]


nan_summary

,title,category,publish_date,author,article_url,content,summary


In [10]:
combined_df=combined_df.dropna()

# Duplicate


In [11]:
data = combined_df.copy()

In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6102 entries, 0 to 6101
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         6102 non-null   object
 1   category      6102 non-null   object
 2   publish_date  6102 non-null   object
 3   author        6102 non-null   object
 4   article_url   6102 non-null   object
 5   content       6102 non-null   object
 6   summary       6102 non-null   object
dtypes: object(7)
memory usage: 333.8+ KB


# Tokenize

Mencoba kode lain agar tokenize lebih mudah di pahami

# Word Embedding

In [13]:
from transformers import AutoTokenizer, AutoModel
import torch

# Memuat IndoBERT dan tokenizer
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1", use_fast=True)
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [14]:
# Fungsi untuk mendapatkan embedding dari IndoBERT
def get_embeddings(input_ids):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
    # Mengambil hidden states dari layer terakhir dan rata-rata embeddingsnya
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

# Fungsi untuk mengonversi tokenized data ke dalam bentuk input_ids dan mendapatkan embedding
def get_input_ids_and_embeddings(data_column):
    input_ids_list = []
    embeddings_list = []

    for text in data_column:
        # Tokenisasi
        tokens = tokenizer(text, padding='max_length', max_length=256, truncation=True, return_tensors="pt")
        input_ids = tokens['input_ids']  # Ambil input_ids
        input_ids_list.append(input_ids)

        # Mendapatkan embeddings
        embedding = get_embeddings(input_ids)
        embeddings_list.append(embedding)

    return input_ids_list, embeddings_list

# Dapatkan input_ids dan embeddings untuk title dan summary
title_input_ids, title_embeddings = get_input_ids_and_embeddings(data['title'])
summary_input_ids, summary_embeddings = get_input_ids_and_embeddings(data['summary'])


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


## Mengubah embeddings menjadi array numpy

In [15]:
# Mengubah embeddings menjadi array numpy agar bisa disimpan dalam DataFrame
title_embeddings_array = np.vstack([embedding.numpy() for embedding in title_embeddings])
summary_embeddings_array = np.vstack([embedding.numpy() for embedding in summary_embeddings])

# Membuat DataFrame baru dengan embeddings numerik
data_embeddings = pd.DataFrame({
    'title_embeddings': list(title_embeddings_array),
    'summary_embeddings': list(summary_embeddings_array)
})

# Cosine Similarity

In [16]:
# hitung cosine similarity pada matrix berita
cosine_sim = cosine_similarity(title_embeddings_array, summary_embeddings_array)
print(cosine_sim)

[[0.87954295 0.95228547 0.40579575 ... 0.903032   0.2260749  0.13818681]
 [0.8706143  0.94533837 0.4029524  ... 0.89482474 0.22230548 0.13384908]
 [0.87885    0.9517845  0.40821669 ... 0.9032042  0.22613734 0.13858905]
 ...
 [0.87099546 0.94564795 0.40239638 ... 0.89665294 0.22158778 0.13032664]
 [0.87574697 0.9515747  0.40250623 ... 0.9023708  0.2237569  0.13417077]
 [0.8734868  0.94632447 0.4063745  ... 0.89720714 0.2249633  0.13594346]]


In [17]:
# membuat dataframe dari varible cosine similarity dengan baris dan kolom title
cosine_sim_df = pd.DataFrame(cosine_sim, index=data['summary'], columns=data['title'])

print(cosine_sim_df.shape)

(6102, 6102)


## Check Similarity Result

In [18]:
# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # The index is a summary, and you need to find the corresponding title
    # to access the similarity value. However, since you have both summaries
    # and titles as indices, you can access the similarity directly using
    # the index and the corresponding title from the 'data' DataFrame.

    title = data.loc[data['summary'] == index, 'title'].iloc[0] # Get the title for the current summary
    similarity_value = cosine_sim_df.loc[index, title]  # Access the similarity using both summary and title

    print(f"Title: {title}")
    print(f"Summary: {index}")
    print(f"Cosine Similarity: {similarity_value}\n")
    print("-" * 50)  # Pemisah antar pasangan

Streaming output truncated to the last 5000 lines.
Cosine Similarity: 0.9643455743789673

--------------------------------------------------
Title: Ngaku Cewek Tulen, Noe Row Mengungkap Alasannya Bergaya Seperti Cowok Ganteng
Summary: Youtuber Indonesia, Noe Row adalah seorang gadis yang tomboy. Tak heran ia selalu dikira cowok ganteng. Padahal ia mengaku perempuan tulen walaupun bergaya rambut pendek dan kacamata bulat. Suaranya yang agak berat, membuatnya juga disangka laki-laki. Padahal Youtuber yang memiliki 1,53 juta subscriber ini memiliki nama asli Nur Firza Hairani. Gadis kelahiran 25 November 1996 asal Medan ini sebelumnya kuliah di Aceh lalu pindah ke salah satu Universitas di Jakarta. Tak hanya itu Noe juga kerap membagikan momen saat bersama pacarnya, bernama Ido. Karena saya suka dengan hal yang berbau simpel," tegasnya. Di luar itu, dalam satu tahun terakhir ini, Noe Row juga disibukkan dengan usaha clothing atau apparel yang digelutinya.
Cosine Similarity: 0.743061006069

In [19]:
# List to store the results
results = []

# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # Mendapatkan judul yang sesuai dengan summary berdasarkan index
    title = data.loc[data['summary'] == index, 'title'].iloc[0]
    similarity_value = cosine_sim_df.loc[index, title]  # Mengakses nilai similarity

    # Menyimpan pasangan dan nilai similarity ke dalam list
    results.append([title, index, similarity_value])

# Membuat DataFrame untuk hasilnya
similarity_df = pd.DataFrame(results, columns=['Title', 'Summary', 'Cosine Similarity'])


In [20]:
similarity_df.head()

,Title,Summary,Cosine Similarity
0,Jokowi Beberkan 'Love-Hate' dengan Surya Paloh...,Presiden Joko Widodo atau Jokowi membeberkan b...,0.879543
1,"Ucap Terima Kasih, Surya Paloh Akui Belajar da...",Ketua Umum Partai NasDem Surya Paloh mengaku b...,0.945338
2,"Di Depan Jokowi, Surya Paloh: Bukan Cuma Masal...",Ketua Umum Partai NasDem Surya Paloh menegaska...,title ...
3,"Jokowi Muncul di Kongres III NasDem Malam Ini,...",Presiden Joko Widodo alias Jokowi menghadiri K...,0.828687
4,"Sebut Pemilu 2024 Paling Brutal, Cak Imin Usul...",Ketua Umum Dewan Pimpinan Pusat Partai Kebangk...,0.300576


## Save Similarity

In [21]:
# Menyimpan DataFrame ke file CSV di lokasi yang ditentukan
similarity_df.to_csv('/content/drive/MyDrive/File/CODES/Similarity/5_Similarity_Suara_Nasional_Base1/similarity_suara_nas_base1.csv', index=False)